# Lección 3.7 — Curvas de aprendizaje: leer la historia del entrenamiento

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección ≈ 70 min · ejercicios finales ≈ 40 min)
> 📋 Requisitos previos: Lecciones 3.3–3.6. Ayuda recordar la división entrenamiento/validación/prueba (Lección 2.5).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Leer una **curva de aprendizaje** (pérdida de entrenamiento y de validación por época) como si fuera un electrocardiograma.
- Diagnosticar a simple vista seis patrones: **sano**, **tasa alta**, **tasa baja**, **sobreajuste**, **subajuste** y **ruido**.
- Elegir **cuándo parar** el entrenamiento con la curva de validación (adelanto de *early stopping*).
- Usar la **otra** curva de aprendizaje (error vs. cantidad de datos) para responder: *¿me sirve conseguir más datos?*

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Una máquina de entrenar (el mismo descenso de gradiente de siempre) que ahora anota en cada época **dos** pérdidas: la de **entrenamiento** y la de **validación**. Y una función para dibujarlas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.seterr(all="ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)

def mse(real, pred):
    return np.mean((pred - real) ** 2)

def entrenar(X_tr, y_tr, X_va, y_va, tasa, epocas, tam_lote=None, semilla=0):
    rng_ent = np.random.default_rng(semilla)
    n, p = X_tr.shape
    tam_lote = tam_lote or n                          # por defecto: batch completo
    w, b = np.zeros(p), 0.0
    hist_tr, hist_va = [mse(y_tr, X_tr @ w + b)], [mse(y_va, X_va @ w + b)]
    for _ in range(epocas):
        orden = rng_ent.permutation(n)
        for inicio in range(0, n, tam_lote):
            lote = orden[inicio:inicio + tam_lote]
            error = X_tr[lote] @ w + b - y_tr[lote]
            w = w - tasa * 2 * X_tr[lote].T @ error / len(lote)
            b = b - tasa * 2 * error.mean()
        hist_tr.append(mse(y_tr, X_tr @ w + b))       # 📘 ¿qué tan bien le va con lo que ESTUDIÓ?
        hist_va.append(mse(y_va, X_va @ w + b))       # 📝 ¿y en el examen de práctica con datos NUEVOS?
    return np.array(hist_tr), np.array(hist_va)

def dibujar_curvas(hist_tr, hist_va, titulo, ax=None, log=True):
    ax = ax or plt.gca()
    ax.plot(hist_tr, lw=2, label="Entrenamiento")
    ax.plot(hist_va, lw=2, ls="--", label="Validación")
    if log: ax.set_yscale("log")
    ax.set_xlabel("Época"); ax.set_ylabel("Pérdida (MSE)"); ax.set_title(titulo); ax.legend(); ax.grid(alpha=.3)

# La cadena de heladerías (Lección 3.4): 800 días para entrenar y 200 para validar
rng = np.random.default_rng(42)
temperatura = rng.uniform(14, 36, 1000)
ventas = 8 * temperatura - 60 + rng.normal(0, 15, 1000)
X = ((temperatura - temperatura.mean()) / temperatura.std()).reshape(-1, 1)
X_tr, X_va, y_tr, y_va = X[:800], X[800:], ventas[:800], ventas[800:]
print("✅ Todo listo")

---
## 1. 📖 La intuición: el electrocardiograma del entrenamiento

Un médico no abre al paciente para saber cómo está su corazón: mira el **electrocardiograma**. Un buen científico de datos tampoco revisa millones de perillas: mira la **curva de aprendizaje**, la historia de la pérdida a lo largo del entrenamiento.

Se dibujan **dos** líneas:
- **Entrenamiento** 📘: la pérdida sobre los datos con los que el modelo **estudia**. Es como medir qué tan bien resuelve los ejercicios del libro que ya leyó.
- **Validación** 📝: la pérdida sobre datos que el modelo **nunca usa para aprender**. Es el **examen de práctica** con preguntas nuevas (Lección 2.5).

La línea de entrenamiento te dice si el modelo **está aprendiendo**. La de validación te dice si lo que aprende **sirve para algo fuera del libro**.

### ✋ Ejercicio 1 — ¿Para qué la segunda línea?
Un estudiante resuelve los 100 ejercicios del libro una y otra vez hasta acertar el 100 %. ¿Garantiza eso que aprobará el examen? ¿Qué te faltaría saber, y cuál de las dos curvas te lo diría?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No lo garantiza: pudo haber **memorizado** las respuestas en vez de entender el método. Te faltaría ver cómo le va con **preguntas que nunca vio**. Eso es justo la curva de **validación**. La de entrenamiento sola puede engañarte: siempre tiende a bajar.
</details>

---
## 2. 📖 El concepto: cómo se ve un entrenamiento sano

Entrenemos la recta de la heladería y miremos sus dos curvas.

In [ ]:
hist_tr, hist_va = entrenar(X_tr, y_tr, X_va, y_va, tasa=0.1, epocas=40)
dibujar_curvas(hist_tr, hist_va, "Un entrenamiento sano"); plt.show()
print(f"Época 40 → entrenamiento: {hist_tr[-1]:.1f} · validación: {hist_va[-1]:.1f}")

Los tres rasgos de una curva sana:
1. **Baja rápido al principio** y luego se **aplana** (el "codo").
2. Las dos curvas **bajan juntas**.
3. Al final, la de validación queda **cerca** de la de entrenamiento (un poco arriba es normal: son datos nuevos).

> 💡 **¿Por qué escala logarítmica?** La pérdida suele bajar de miles a decenas. En escala normal, todo lo interesante del final queda aplastado contra el suelo. En escala log cada "escalón" de ×10 ocupa lo mismo.

### ✋ Ejercicio 2 — Lee la curva
1. ¿En qué época, aproximadamente, ya no vale la pena seguir entrenando?
2. Al final, ¿la validación queda por encima o por debajo del entrenamiento? ¿Es grave?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Hacia la época **20–25** las dos curvas ya están planas. Seguir hasta 40 no cambia nada.
2. La validación termina **un poquito por encima** (≈ 235 contra ≈ 230). Es lo normal y no es grave: el modelo rinde casi igual con días que nunca vio. Lo importante es que estén **cerca**. (Si la validación quedara **muy** por debajo de forma sistemática, sospecharías de algo raro, como datos repetidos entre los dos conjuntos.)
</details>

---
## 3. 📖 La galería de diagnósticos

Ahora viene lo más útil de la lección: aprender a reconocer los **patrones típicos** a primera vista. Para cada uno, primero **predice** y luego ejecuta.

### 🔮 Patrón A — Tasa demasiado alta
¿Cómo crees que se verán las curvas con tasa **1.02** (justo por encima del precipicio de la Lección 3.4)? ¿Y con **0.97** (justo por debajo)?

✍️ *Mi dibujo / descripción:*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
dibujar_curvas(*entrenar(X_tr, y_tr, X_va, y_va, tasa=1.02, epocas=40), "A1. Tasa 1.02: diverge", ax=a1)
dibujar_curvas(*entrenar(X_tr, y_tr, X_va, y_va, tasa=0.97, epocas=40), "A2. Tasa 0.97: zigzag", ax=a2)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **A1 (1.02):** las dos curvas **suben** sin parar: el excursionista salió volando. Con tasas aún mayores, la pérdida llega a `inf` o `nan` en pocas épocas.
- **A2 (0.97):** las curvas bajan, pero de forma **dentada**, con un patrón de sube-y-baja de época en época, y tardan mucho más que la curva sana: es el **zigzag** entre las paredes del valle.

🩺 **Diagnóstico:** tasa demasiado alta → **dividir la tasa entre 3–10**.
</details>

### 🔮 Patrón B — Tasa demasiado baja
¿Y con tasa **0.002**?

✍️ *Tu respuesta:*

In [ ]:
dibujar_curvas(*entrenar(X_tr, y_tr, X_va, y_va, tasa=0.002, epocas=40), "B. Tasa 0.002"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos curvas bajan **casi en línea recta, sin codo**, y al terminar **siguen bajando**: el modelo no ha terminado de aprender (≈ 16,000 contra ≈ 230 del entrenamiento sano). Es como interrumpir a alguien a mitad de una frase.

🩺 **Diagnóstico:** tasa demasiado baja o pocas épocas → **subir la tasa** (barrido, Lección 3.4) o entrenar más.
</details>

### 🔮 Patrón C — Sobreajuste
Una tienda en línea quiere predecir el gasto de sus clientes. Tiene **solo 40 clientes** para entrenar, pero **30 columnas** de información sobre cada uno, y en realidad solo 3 importan. Un modelo con 31 perillas y 40 ejemplos… ¿qué crees que pasará con la curva de **validación** si entrenamos muchas épocas?

✍️ *Tu respuesta:*

In [ ]:
rng_s = np.random.default_rng(0)
w_real = np.zeros(30); w_real[:3] = [3, -2, 1.5]            # solo 3 de las 30 columnas importan
X_s_tr = rng_s.normal(size=(40, 30));  y_s_tr = X_s_tr @ w_real + rng_s.normal(0, 2, 40)
X_s_va = rng_s.normal(size=(500, 30)); y_s_va = X_s_va @ w_real + rng_s.normal(0, 2, 500)

tr_s, va_s = entrenar(X_s_tr, y_s_tr, X_s_va, y_s_va, tasa=0.01, epocas=3000)
dibujar_curvas(tr_s, va_s, "C. 40 clientes, 30 columnas")
mejor_epoca = np.argmin(va_s)
plt.axvline(mejor_epoca, color="green", ls=":", label=f"mínimo de validación (época {mejor_epoca})"); plt.legend(); plt.show()
print(f"Época {mejor_epoca:>4} → entrenamiento: {tr_s[mejor_epoca]:.2f} · validación: {va_s[mejor_epoca]:.2f}")
print(f"Época 3000 → entrenamiento: {tr_s[-1]:.2f} · validación: {va_s[-1]:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La curva de entrenamiento **baja y baja** (llega a ≈ 0.30), pero la de validación baja solo al principio, toca fondo en la época **168** (≈ 9.1) y luego **sube** hasta ≈ 19.7: más del doble.

Es la firma inconfundible del **sobreajuste** (Lección 4.4): después de aprender lo importante, el modelo empezó a usar sus 31 perillas para **memorizar el ruido** de los 40 clientes. Cada época extra lo hace mejor en el libro y peor en el examen.

🩺 **Diagnóstico:** sobreajuste → **parar antes** (la línea verde: *early stopping*, Sección 4), **más datos**, **menos columnas** o **regularización** (Lección 7.1).
</details>

### 🔮 Patrón D — Subajuste
Volvamos a la cafetería de la Lección 3.1: los cafés helados suben con el calor… hasta que hace demasiado calor y bajan. ¿Qué crees que mostrarán las curvas si intentamos ajustar una **recta**?

✍️ *Tu respuesta:*

In [ ]:
rng_c = np.random.default_rng(2)
temp_c = rng_c.uniform(10, 40, 400)
cafes = -0.5 * (temp_c - 28) ** 2 + 120 + rng_c.normal(0, 8, 400)
t_esc = ((temp_c - temp_c.mean()) / temp_c.std()).reshape(-1, 1)

tr_c, va_c = entrenar(t_esc[:300], cafes[:300], t_esc[300:], cafes[300:], tasa=0.1, epocas=60)
dibujar_curvas(tr_c, va_c, "D. Una recta para datos curvos"); plt.show()
print(f"Final → entrenamiento: {tr_c[-1]:.0f} · validación: {va_c[-1]:.0f} · (el ruido de los datos es ≈ 8² = 64)")

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos curvas bajan y se **aplanan juntas**, pero **muy arriba**: ≈ 1,100–1,200, cuando el ruido natural de los datos es ≈ 64. No hay brecha entre ellas, así que **no** es sobreajuste: es que el modelo **no tiene capacidad** para representar la curva, entrene lo que entrene.

🩺 **Diagnóstico:** subajuste → **un modelo más flexible** o **mejores características**. Más épocas o más datos **no** ayudan. Compruébalo en el ejercicio siguiente.
</details>

### ✏️ Ejercicio 3 — Cura el subajuste
Añade una segunda columna: la **temperatura al cuadrado** (escalada). Así la "recta" puede doblarse en una parábola (Lección 3.1). ¿Cuánto baja la pérdida de validación?

In [ ]:
t2 = t_esc[:, 0] ** 2
t2_esc = (t2 - t2.mean()) / t2.std()
X_c2 = np.column_stack([t_esc[:, 0], t2_esc])        # ✏️ prueba a quitar la segunda columna

tr_c2, va_c2 = entrenar(X_c2[:300], cafes[:300], X_c2[300:], cafes[300:], tasa=0.1, epocas=60)
dibujar_curvas(tr_c2, va_c2, "D'. Con temperatura²"); plt.show()
print(f"Final → entrenamiento: {tr_c2[-1]:.0f} · validación: {va_c2[-1]:.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Ambas curvas bajan hasta ≈ **60–70**, prácticamente el nivel del ruido (64). Darle al modelo la **forma** correcta resolvió lo que ninguna cantidad de épocas podía resolver.
</details>

### 🔮 Patrón E — Curva ruidosa
Entrenamos la heladería con **SGD** (lote = 1, Lección 3.5) y tasa 0.05. Esta vez dibujamos la pérdida **después de cada paso**, no de cada época. ¿Cómo se verá?

✍️ *Tu respuesta:*

In [ ]:
rng_e = np.random.default_rng(0)
w, b, perd_paso = 0.0, 0.0, []
for epoca in range(3):
    for i in rng_e.permutation(800):
        error = X_tr[i, 0] * w + b - y_tr[i]
        w, b = w - 0.05 * 2 * error * X_tr[i, 0], b - 0.05 * 2 * error
        perd_paso.append(mse(y_va, X_va[:, 0] * w + b))
perd_paso = np.array(perd_paso)

VENTANA = 50          # ✏️ prueba 1, 10, 200
suavizada = np.convolve(perd_paso, np.ones(VENTANA) / VENTANA, mode="valid")   # promedio móvil
plt.plot(perd_paso, alpha=.35, label="Pérdida de validación en cada paso")
plt.plot(np.arange(VENTANA - 1, len(perd_paso)), suavizada, lw=2, color="k", label=f"Promedio móvil ({VENTANA} pasos)")
plt.yscale("log"); plt.ylim(150, 3000); plt.xlabel("Paso"); plt.ylabel("Pérdida"); plt.title("E. Curva ruidosa (SGD)"); plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La curva cruda **tiembla** muchísimo: cada paso sigue la opinión de un solo día (Lección 3.5). Leerla así es como leer un electrocardiograma con interferencia.

El truco es **suavizarla** con un **promedio móvil**: cada punto es el promedio de los últimos `VENTANA` pasos. La línea negra deja ver la tendencia: baja rápido y se estabiliza alrededor de ≈ 250–280 (un poco por encima del ≈ 235 del entrenamiento sano: es el precio del ruido con tasa fija, Lección 3.5). Herramientas como TensorBoard tienen un control de "suavizado" que hace exactamente esto.

🩺 **Diagnóstico:** ruido normal de SGD/mini-batch → **suaviza para leer la tendencia**. Si el ruido es tan grande que no deja ver nada, sube el lote o baja la tasa.
</details>

---
## 4. 📖 ¿Cuándo parar? La idea del *early stopping*

En el Patrón C viste que la validación toca fondo y luego sube. La solución más natural: **vigilar la curva de validación y parar cuando deja de mejorar**. Para no reaccionar ante un tropiezo pasajero, se da un margen de **paciencia**: "si en `PACIENCIA` épocas seguidas no mejora el récord, me detengo y me quedo con la mejor época".

Esto se llama **parada temprana** (*early stopping*) y lo verás a fondo en la **Lección 7.2**.

### ✏️ Ejercicio 4 — Ponle paciencia
Aplica la regla a las curvas del Patrón C. Prueba `PACIENCIA` = 5, 50 y 500.

In [ ]:
PACIENCIA = 50        # ✏️

mejor, mejor_epoca, sin_mejorar = np.inf, 0, 0
for epoca, perdida_va in enumerate(va_s):
    if perdida_va < mejor:
        mejor, mejor_epoca, sin_mejorar = perdida_va, epoca, 0
    else:
        sin_mejorar += 1
        if sin_mejorar >= PACIENCIA:
            break
print(f"Paciencia {PACIENCIA}: me detengo en la época {epoca} y me quedo con la época {mejor_epoca} "
      f"(validación = {mejor:.2f}). Ahorro {3000 - epoca} épocas de cómputo.")

<details><summary><b>👀 Ver respuesta</b></summary>

Esta curva es muy suave, así que con **cualquiera** de las tres paciencias se queda con la misma época, la **168** (validación ≈ 9.10). Lo que cambia es **cuándo** se detiene: con paciencia 5, en la época 173; con 50, en la 218; con 500, en la 668. Más paciencia = más cómputo gastado.

En curvas **ruidosas** (como la del Patrón E) la paciencia importa mucho más: con un valor muy bajo te detendrías ante el primer tropiezo del ruido, demasiado pronto. Un punto de partida común: 5–20 épocas.
</details>

---
## 5. 📖 La otra curva de aprendizaje: ¿me sirve conseguir más datos?

Hay una **segunda** gráfica que también se llama "curva de aprendizaje": el error en función de **cuántos datos** usas para entrenar (en vez de cuántas épocas). Responde una pregunta carísima: *¿vale la pena pagar por más datos?*

scikit-learn la calcula con `learning_curve` (entrena el modelo con 10 %, 20 %, … del conjunto, con validación cruzada, Lección 4.6). Comparemos dos casos:
- **La tienda del Patrón C** (30 columnas, sobreajusta con pocos datos).
- **La cafetería con una recta** (Patrón D, subajusta).

### 🔮 Ejercicio 5
¿En cuál de los dos casos crees que **más datos** mejorarán mucho la validación?

✍️ *Tu respuesta:*

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LinearRegression

rng_lc = np.random.default_rng(1)
X_tienda = rng_lc.normal(size=(2000, 30)); y_tienda = X_tienda @ w_real + rng_lc.normal(0, 2, 2000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (Xc, yc, titulo) in zip(axes, [(X_tienda, y_tienda, "Tienda (30 columnas)"), (t_esc, cafes, "Cafetería con una recta")]):
    tamanos, err_tr, err_va = learning_curve(LinearRegression(), Xc, yc, cv=5, scoring="neg_mean_squared_error",
                                             train_sizes=np.linspace(0.025, 1.0, 8), shuffle=True, random_state=0)
    ax.plot(tamanos, -err_tr.mean(axis=1), "o-", label="Entrenamiento")
    ax.plot(tamanos, -err_va.mean(axis=1), "s--", label="Validación")
    ax.set_yscale("log"); ax.set_xlabel("Ejemplos de entrenamiento"); ax.set_ylabel("MSE"); ax.set_title(titulo); ax.legend(); ax.grid(alpha=.3)
    print(f"{titulo:<24} con {tamanos[0]:>4} ejemplos → validación {-err_va.mean(axis=1)[0]:8.1f} · con {tamanos[-1]:>4} → {-err_va.mean(axis=1)[-1]:8.1f}")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Tienda:** con 40 ejemplos la brecha es enorme (sobreajuste: validación ≈ 21), y al añadir datos la validación **baja mucho** y se acerca al entrenamiento (≈ 4.0 con 1,600 ejemplos, el ruido natural 2² = 4). Aquí **más datos sí ayudan**.
- **Cafetería con recta:** las dos curvas se juntan **muy pronto** y se quedan **arriba** (≈ 1,200 con 320 ejemplos). Más datos **no** ayudan: el problema es el modelo, no la cantidad de ejemplos.

📌 Regla: **brecha grande que se cierra al añadir datos → consigue más datos. Curvas juntas y altas → cambia el modelo o las características.**
</details>

---
## 6. 🩺 Tabla de diagnóstico

| Lo que ves en la curva | Diagnóstico | Qué probar |
|---|---|---|
| Ambas bajan con codo y se aplanan juntas, cerca | ✅ Sano | Parar cuando se aplane la validación |
| Suben, o llegan a `inf` / `nan` | Tasa **muy alta** | Dividir la tasa entre 3–10 |
| Bajan en zigzag, dentadas | Tasa **algo alta** | Bajar un poco la tasa; subir el lote |
| Bajan casi en línea recta y siguen bajando al final | Tasa **baja** o **pocas épocas** | Subir la tasa; más épocas |
| Plana mucho tiempo y luego cae | **Meseta** (Lección 3.6) | Paciencia; tasa mayor; momento |
| Entrenamiento baja, validación baja y luego **sube** | **Sobreajuste** | *Early stopping*; más datos; regularización (7.1); modelo más simple |
| Ambas planas y **altas**, juntas | **Subajuste** | Modelo más flexible; mejores características |
| Tiembla mucho | Ruido de lotes chicos | Suavizar para leer; subir el lote o bajar la tasa |
| Validación **mucho mejor** que entrenamiento, siempre | Algo raro: datos repetidos o fuga entre conjuntos, o regularización que solo actúa al entrenar (*dropout*, Lección 9.8) | Revisar cómo se dividieron los datos (Lección 2.6) |

## 7. 📖 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Mirar solo la curva de entrenamiento | Siempre tiende a bajar: no detecta el sobreajuste | Dibuja **siempre** las dos curvas |
| Usar el conjunto de **prueba** como validación para decidir cuándo parar | Lo "gastas": tu resultado final deja de ser honesto | Validación para decidir; prueba **una vez** al final (Lección 4.6) |
| Escala lineal en el eje de la pérdida | Esconde lo que pasa al final del entrenamiento | Usa escala logarítmica |
| Asustarse por el ruido de una curva con lotes chicos | Es normal en SGD/mini-batch | Suaviza con promedio móvil |
| Concluir "más datos lo arreglará" sin mirar la curva vs. datos | Si hay subajuste, no ayudará | Mira si hay brecha que se cierra |

---
## 8. 🧠 Resumen en 5 puntos
1. La **curva de aprendizaje** es el electrocardiograma del entrenamiento: pérdida de **entrenamiento** y de **validación** por época.
2. Sana = bajan con codo y se aplanan **juntas**. Las tasas mal elegidas dejan huellas claras: **sube/zigzag** (alta) o **recta sin codo** (baja).
3. **Sobreajuste** = la validación toca fondo y luego **sube**. **Subajuste** = ambas planas y **altas**.
4. Para decidir cuándo parar, vigila la **validación** con algo de **paciencia** (*early stopping*, Lección 7.2).
5. La curva de error **vs. cantidad de datos** dice si conseguir más datos vale la pena: sí cuando hay brecha que se cierra; no cuando ambas están altas y juntas.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso con curvas misteriosas 🕵️
Ejecuta la celda: dibuja **cuatro** entrenamientos de los que no sabes nada. Diagnostica cada uno con la tabla de la Sección 6.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
dibujar_curvas(*entrenar(X_s_tr, y_s_tr, X_s_va, y_s_va, tasa=0.005, epocas=2500), "Misterio 1", ax=axes[0, 0])
dibujar_curvas(*entrenar(X_tr, y_tr, X_va, y_va, tasa=1.005, epocas=60), "Misterio 2", ax=axes[0, 1])
dibujar_curvas(*entrenar(t_esc[:300], cafes[:300], t_esc[300:], cafes[300:], tasa=0.3, epocas=60), "Misterio 3", ax=axes[1, 0])
dibujar_curvas(*entrenar(X_tr, y_tr, X_va, y_va, tasa=0.005, epocas=60), "Misterio 4", ax=axes[1, 1])
plt.tight_layout(); plt.show()

✍️ *Mis diagnósticos:* 1 → ___ · 2 → ___ · 3 → ___ · 4 → ___

Y responde además:

**5.** Si solo pudieras mirar **una** de las dos curvas para detectar sobreajuste, ¿cuál sería?
- a) Entrenamiento · b) Validación · c) Da igual

**6.** ¿Cuál de estos remedios **no** sirve contra el subajuste?
- a) Un modelo más flexible · b) Mejores características · c) Parar antes (*early stopping*) · d) Quitar regularización

<details><summary><b>👀 Ver respuestas</b></summary>

1. **Sobreajuste**: la validación baja, toca fondo y luego sube, mientras el entrenamiento sigue bajando (la tienda de 40 clientes).
2. **Tasa demasiado alta**: zigzag que va empeorando hasta subir (justo por encima del precipicio).
3. **Subajuste**: bajan rápido, se aplanan juntas y muy arriba (una recta para datos curvos).
4. **Tasa demasiado baja**: bajan casi en línea recta, sin codo, y siguen bajando al terminar.
5. **b)** La de validación: el sobreajuste es justo que la validación empeore aunque el entrenamiento mejore.
6. **c)** Parar antes empeora el subajuste: el modelo aprende todavía menos.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Tu equipo entrena un modelo carísimo durante 3 días. Al final, la validación es peor que en el día 1. ¿Qué deberían haber hecho, y qué les dirías para la próxima vez?

**B.** Tu jefe quiere gastar $50,000 en etiquetar más datos. Tus curvas de entrenamiento y validación están juntas, planas y con un error alto. ¿Le recomiendas gastar ese dinero? ¿Qué propones en cambio?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Deberían haber **vigilado la curva de validación** durante el entrenamiento y **guardado el modelo de la mejor época** (*early stopping* con *checkpoints*, Lección 7.2). Para la próxima: dibujar las dos curvas en vivo, definir una paciencia y guardar siempre el mejor modelo según validación, no el último.

**B.** **No**: curvas juntas, planas y altas = **subajuste**. Más datos no lo arreglarán (Sección 5). Propón primero un modelo más flexible o mejores características (Lecciones 2.4 y Módulo 6), y gasta en datos solo si después aparece una brecha que se cierra al añadir ejemplos.
</details>

### 9.3 🧩 Práctica integradora: el doctor de entrenamientos 🩺

Un hospital te pide un modelo que estime los **días de hospitalización** de un paciente a partir de **30 mediciones** de su ingreso (ya escaladas). Tienen pocos pacientes. Tu misión: entrenar, diagnosticar con las curvas y arreglar lo que salga mal.

**Paso 1 ▶️ — Los datos**

In [ ]:
rng_h = np.random.default_rng(21)
w_h = np.zeros(30); w_h[:5] = [3.0, -2.0, 1.5, 1.0, -1.0]          # solo 5 mediciones importan de verdad
def pacientes(n):
    Xp = rng_h.normal(size=(n, 30))
    return Xp, 6 + Xp @ w_h + rng_h.normal(0, 2.0, n)
X_h_tr, y_h_tr = pacientes(50)      # 50 pacientes para entrenar
X_h_va, y_h_va = pacientes(300)     # 300 para validar
X_h_te, y_h_te = pacientes(300)     # 300 para la prueba final (¡no tocar hasta el Paso 6!)
print("Pérdida si predices siempre el promedio (línea base, Lección 4.7):", round(mse(y_h_va, np.full(300, y_h_tr.mean())), 2))

**Paso 2 🔮 — Primer intento.** Un compañero entrenó con tasa **0.5** durante 300 épocas. Predice qué verás y luego ejecuta.

✍️ *Tu respuesta:*

In [ ]:
tr_h, va_h = entrenar(X_h_tr, y_h_tr, X_h_va, y_h_va, tasa=0.5, epocas=300)
dibujar_curvas(tr_h, va_h, "Intento 1: tasa 0.5"); plt.show()
print("Últimas pérdidas de validación:", va_h[-3:])

<details><summary><b>👀 Ver respuesta</b></summary>

Las curvas **suben** sin control (≈ 10¹¹⁸ al final): **tasa demasiado alta**. Con 30 columnas y solo 50 pacientes, el valle es más empinado que el de la heladería y 0.5 ya es demasiado.
</details>

**Paso 3 ✏️ — Arregla la tasa.** Haz un mini barrido: prueba tasas de 0.3, 0.1, 0.03 y 0.01 durante 300 épocas y quédate con la que tenga **menor validación mínima**. Anota en qué época ocurre ese mínimo.

In [ ]:
for tasa in [0.3, 0.1, 0.03, 0.01]:           # ✏️
    tr_h, va_h = entrenar(X_h_tr, y_h_tr, X_h_va, y_h_va, tasa=tasa, epocas=300)
    print(f"tasa {tasa:<5} → validación mínima {np.nanmin(va_h):10.3f} en la época {np.nanargmin(va_h):>3} · validación final {va_h[-1]:10.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Las cuatro tasas llegan a una validación mínima casi idéntica (**≈ 7.7**), pero en épocas distintas: cuanto más baja la tasa, **más tarde** (épocas 10, 32, 109 y 300).
- Con 0.3, 0.1 y 0.03, la validación **final** (época 300) es **peor** que la mínima (12.5, 11.0 y 8.5): ahí hay algo más… (Paso 4). Con 0.01 todavía no llegó a ese punto.

**0.1** es un buen equilibrio: rápida, estable y con una curva fácil de leer.
</details>

**Paso 4 🔮 — Mira la curva completa.** Dibuja las curvas con tasa 0.1 y 300 épocas. ¿Qué segundo problema ves?

✍️ *Tu respuesta:*

In [ ]:
TASA_H = 0.1        # ✏️ tu tasa elegida
tr_h, va_h = entrenar(X_h_tr, y_h_tr, X_h_va, y_h_va, tasa=TASA_H, epocas=300)
dibujar_curvas(tr_h, va_h, f"Intento 2: tasa {TASA_H}"); plt.show()
print(f"Mínimo de validación: {va_h.min():.2f} en la época {va_h.argmin()} · final: {va_h[-1]:.2f} · entrenamiento final: {tr_h[-1]:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Sobreajuste**: el entrenamiento sigue bajando (≈ 0.85) mientras la validación toca fondo en la época **32** (≈ 7.7) y luego **sube** (≈ 11.0 al final). Con 31 perillas y 50 pacientes, el modelo termina memorizando.
</details>

**Paso 5 ✏️ — ¿Más pacientes ayudarían?** El hospital puede conseguir más historiales, pero cuesta. Entrena con 50, 100, 200 y 400 pacientes (misma tasa y 300 épocas) y mira la validación **final**.

In [ ]:
for n in [50, 100, 200, 400]:          # ✏️
    X_n, y_n = pacientes(n)
    tr_n, va_n = entrenar(X_n, y_n, X_h_va, y_h_va, tasa=TASA_H, epocas=300)
    print(f"{n:>3} pacientes → entrenamiento final {tr_n[-1]:.2f} · validación final {va_n[-1]:.2f} · brecha {va_n[-1] - tr_n[-1]:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con más pacientes la **brecha se cierra** (de ≈ 9.7 a ≈ 0.2) y la validación final **baja** (de ≈ 11.9 a ≈ 4.1, casi el ruido natural 2² = 4). El sobreajuste desaparece casi por completo con 400. **Más datos sí ayudan aquí**, justo el caso "brecha que se cierra" de la Sección 5.

(Ojo: cada llamada a `pacientes()` genera pacientes nuevos, así que tus números exactos pueden cambiar un poco si ejecutas la celda dos veces.)
</details>

**Paso 6 ✋ — Decisión final y examen.** Supón que el hospital **no** puede conseguir más datos. Usa *early stopping*: entrena con los 50 pacientes, tu tasa y **las épocas de la mejor validación** del Paso 4. Solo entonces evalúa **una vez** en el conjunto de prueba.

In [ ]:
EPOCAS_FINALES = int(va_h.argmin())        # ✏️ la mejor época del Paso 4
tr_f, te_f = entrenar(X_h_tr, y_h_tr, X_h_te, y_h_te, tasa=TASA_H, epocas=EPOCAS_FINALES)
tr_300, te_300 = entrenar(X_h_tr, y_h_tr, X_h_te, y_h_te, tasa=TASA_H, epocas=300)
print(f"Con early stopping ({EPOCAS_FINALES} épocas) → MSE en prueba = {te_f[-1]:.2f} · error típico ≈ {np.sqrt(te_f[-1]):.1f} días")
print(f"Sin parar (300 épocas)              → MSE en prueba = {te_300[-1]:.2f} · error típico ≈ {np.sqrt(te_300[-1]):.1f} días")
print(f"Línea base (predecir el promedio)   → MSE en prueba = {mse(y_h_te, np.full(300, y_h_tr.mean())):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con *early stopping* (32 épocas) el MSE en prueba es ≈ **7.8** (≈ 2.8 días de error típico), contra ≈ **13.3** (≈ 3.6 días) si entrenas las 300 épocas y ≈ **19.7** (≈ 4.4 días) de la línea base. Parar a tiempo mejoró el modelo **sin cambiar nada más**.

Para el informe al hospital: *"El modelo estima la estancia con un error típico de casi 3 días, frente a más de 4 días si usáramos el promedio de siempre. Con 4 veces más historiales, las pruebas indican que el error podría bajar a unos 2 días."*
</details>

---
## 📚 Para profundizar
- 📘 **Lectura:** Andrej Karpathy, *"A Recipe for Training Neural Networks"* (blog, 2019): cómo mira las curvas un experto.
- 📘 **Documentación:** scikit-learn, *Validation curves and learning curves* (`learning_curve`, `LearningCurveDisplay`).
- 🎮 **Herramienta:** TensorBoard (o Weights & Biases): dibujan las curvas en vivo mientras entrenas, con control de suavizado.

---
⬅️ **Anterior:** [3.6 Mínimos locales, mesetas y por qué no siempre importan](3.6-minimos-locales-y-mesetas.ipynb) · ➡️ **Siguiente:** [3.8 Proyecto del módulo: Entrenador a mano 🩺⚙️](3.8-proyecto-del-modulo.md)